# Running Hist2ST experiments on Expanse

This guide explains why and how to move Hist2ST experiments from an interactive laptop or Colab workflow to SDSC's Expanse high-performance computing (HPC) system. It uses the course's week 08 Expanse slides as background and the current SDSC Expanse User Guide for operational details that may change.

**In brief:** prepare code and data, test a small run on an allocated compute node, submit the full experiment to Slurm, monitor its job ID, and copy important results to backed-up storage. Expanse is a remote cluster: submitting a job does not run it on your laptop.


## 1. Why use Expanse?

Hist2ST works with many large H&E image patches and gene-expression arrays. Extracting foundation-model embeddings and training neural models can need more GPU memory, RAM, disk throughput, or runtime than a laptop provides. Expanse provides shared CPU/GPU compute nodes and a scheduler that queues work until the requested resources are available.

The login node is for connecting, editing small files, checking jobs, and preparing submissions. Long or resource-heavy experiments belong on allocated compute nodes. Slurm records the requested resources and job state, lets you run without keeping a terminal open, and produces logs that help reproduce or debug an experiment.

For this project, a reasonable first mapping is:

- **Notebook 01, data preparation:** CPU job for filtering metadata and building aligned tables; data downloads may need to be staged separately.
- **Notebook 02, GenBio-PathFM embeddings:** GPU job; run a one-patch smoke test before processing every section.
- **Notebook 03, PCA and ridge evaluation:** CPU job; choose memory and runtime based on a small test.
- **Transformer experiments:** GPU job, with checkpoints written to persistent project storage during training.

This is why we use the cluster: queue the expensive work where the CPU, GPU, memory, and storage are, while keeping analysis scripts and logs organized and recoverable.


## 2. What the course slides say, and what to verify

The attached **BDD Expanse Cluster Usage week 08** slides (dated October 20, 2026) introduce ACCESS accounts, SSH/SCP, storage areas, partitions, `srun`, `sbatch`, job monitoring, modules, and containers. Their examples include the account label `jhu159` and a demo directory containing `canc`. Those are examples from the slides; do not copy the demo's user-specific path or assume that `jhu159` is your valid allocation.

Before submitting, check your own CPU and GPU allocations, Slurm account name, partition access, storage path, and software instructions with the course staff and current SDSC documentation. The current Expanse guide may differ from older slide examples. In particular, current guidance says that jobs using Lustre must request the `lustre` constraint, and that Lustre project/scratch storage is not a backup archive.


## 3. Get access and connect

You need an ACCESS account and an active allocation that includes the Expanse resource you plan to use. The course slides say to use your JHU email and ask the instructor to add you to the course project. Follow the course's current enrollment instructions and complete Expanse's required two-factor authentication.

From Windows PowerShell, Terminal, or MobaXterm, connect with SSH. Replace `USERNAME` with your Expanse login name:

```powershell
ssh USERNAME@login.expanse.sdsc.edu
```

On Expanse, check the account, current directory, and available space. Use the project identifier reported for your own allocation in the `--account` option of Slurm jobs. Check GPU allocation separately if you plan to use GPUs.

```bash
module load sdsc
expanse-client user -r expanse
expanse-client user -r expanse_gpu
pwd
df -h
```

If the GPU resource name is unavailable in your account, ask the course staff which allocation and partition you should use. The Slurm account/project name is not necessarily your Expanse username.


## 4. Put code, data, and results in the right place

Expanse has several file systems with different purposes:

- **`$HOME`:** source code and small scripts/configuration. It has a quota and is not intended for high-throughput experiment I/O; do not run full experiments from here.
- **Project Lustre:** active shared code/data/results for an allocation. Use the exact project directory given by your PI/course or shown in your account instructions. It is not backed up; current SDSC policy says project data is purged 90 days after the allocation expires.
- **Global scratch Lustre:** temporary working data. Scratch files are purged 90 days after creation, so move needed results out in time.
- **Node-local `/scratch/$USER/job_$SLURM_JOB_ID`:** fast temporary storage available only while the job is running. Copy anything you need back before the job ends.

Keep raw HEST inputs in the approved project or data location, model outputs/checkpoints in a clearly named run directory, and disposable intermediates in scratch. Back up important outputs somewhere outside Expanse Lustre. The current SDSC guide requires `#SBATCH --constraint="lustre"` for jobs that need Lustre.

Create a project workspace using the path assigned to you (the path below is a placeholder):

```bash
PROJECT_DIR=/expanse/lustre/projects/REPLACE_WITH_ASSIGNED_PATH/hist2st
mkdir -p "$PROJECT_DIR"/{repo,data,runs,logs}
```

Do not paste a path from the instructor's demo unless it is explicitly the path assigned to your account.


## 5. Transfer code and stage input data

For a small code transfer from Windows, first create the destination directory on Expanse, then copy only the source and configuration files you need. Replace the username and destination with your own values.

```powershell
ssh USERNAME@login.expanse.sdsc.edu "mkdir -p /expanse/lustre/projects/REPLACE_WITH_ASSIGNED_PATH/hist2st/repo"
scp -r .\src .\notebooks .\jobs .\configs .\requirements.txt USERNAME@login.expanse.sdsc.edu:/expanse/lustre/projects/REPLACE_WITH_ASSIGNED_PATH/hist2st/repo/
```

For large HEST data, do not push repeated or large transfers through a login node. Use the Expanse User Portal, FileZilla/SFTP for modest transfers, or the large-data transfer method approved by SDSC/course staff. Check that a transfer completed before scheduling a job. Keep `.env`, access tokens, private keys, and other credentials out of Git and out of the copied project tree.

The Hist2ST notebooks need a consistent data root. Before batch execution, set a string `DATA_DIR_NAME` from the environment, then derive a `DATA_DIR_PATH` with `Path`, rather than writing to a laptop-relative `../data` path:

```python
from pathlib import Path

DATA_DIR_NAME = "/expanse/lustre/projects/REPLACE_WITH_ASSIGNED_PATH/hist2st/data"  # set the shared data path here
DATA_DIR_PATH = Path(DATA_DIR_NAME).expanduser()
RUN_DIR = Path("/expanse/lustre/projects/REPLACE_WITH_ASSIGNED_PATH/hist2st/runs/REPLACE_WITH_RUN_NAME").expanduser()
RUN_DIR.mkdir(parents=True, exist_ok=True)
```

Use the same explicit data path in each notebook stage. Environment variables belong in scripts and batch-job configuration. Stage gated Hugging Face files using an approved route; compute nodes may not have the outbound network access or interactive authentication that a laptop has.


## 6. Choose a software environment

Expanse uses environment modules. Inspect available software instead of guessing version names. CPU and GPU module trees are selected separately; do not load both at once.

```bash
module purge
module load cpu       # use this for CPU work; use `module load gpu` for GPU work
module spider python
module spider pytorch
module spider singularity
module list
```

The week 08 slides recommend modules or a Singularity container and warn that user-installed packages can create many small files. Follow the course's supported environment recipe. A container is useful when you need a fixed set of Python/PyTorch dependencies; a module-based environment can be simpler when the needed versions are provided. Check the current Expanse instructions for the exact module/container names.

The repository's `requirements.txt` installs the project's default PyTorch build; do not assume it provides CUDA acceleration on Expanse. For a GPU run, use the course/SDSC-approved GPU-capable PyTorch environment and verify it on an allocated GPU node:

```bash
python -c 'import torch; print(torch.__version__); print("CUDA available:", torch.cuda.is_available())'
nvidia-smi
```

Install dependencies once when preparing the environment, not on every job. Record the module list or container image/version with the run.


## 7. Make the current notebooks batch-safe

Notebooks 01-03 are designed for interactive use. Before submitting them through Jupyter's `nbconvert`, adapt them for a non-interactive cluster job:

1. Install dependencies into the chosen environment ahead of time; remove or skip notebook `%pip install` cells in the batch run.
2. Remove or skip Colab Drive mounting. Set `DATA_DIR_NAME` and `RUN_DIR` explicitly in the notebook settings cell as shown above.
3. Replace interactive `login()` prompts with a course-approved, non-interactive authentication method. Never write a Hugging Face token into a notebook, batch script, repository, or log. If a required file can be staged in advance, skip its download cell.
4. Make output directories explicit and unique per run. Do not overwrite a previous run's metrics or checkpoints.
5. Save the executed notebook, config/parameters, Slurm log, and model outputs under the run directory.

The notebooks depend on one another: Notebook 02 needs Notebook 01's aligned data, and Notebook 03 needs both the prepared expression and embeddings. Submit them as separate jobs because the embedding step needs a GPU while the evaluation step is CPU-oriented. Use `nbconvert` only after these edits and after verifying that the notebook's kernel can see the prepared Python environment.


## 8. Start with a short interactive test

Use an interactive allocation to check imports, paths, and a small input before committing to a long batch run. The example requests a small CPU debug allocation; confirm your project and current partition limits first. Add the Lustre constraint if the test reads or writes Lustre.

```bash
srun --partition=debug --pty --account=YOUR_PROJECT \
  --constraint=lustre --nodes=1 --ntasks=1 --cpus-per-task=4 \
  --mem=8G --time=00:20:00 --wait=0 --export=ALL /bin/bash
```

When the prompt changes to a compute-node name, check the environment and run a small smoke test. Exit the shell when finished; the allocated resources are then released. Do not use an interactive allocation as an unattended, multi-hour training job.


## 9. Submit a CPU experiment with `sbatch`

A batch script is a recipe for Slurm: it names the job, requests resources, prepares the environment, and launches the program. Here is a **template** for an adapted CPU notebook such as Notebook 01 or Notebook 03. Replace the project path, account, environment setup, and resource requests before use. Create `logs/` before calling `sbatch`, because Slurm opens the output file when the job starts.

```bash
#!/bin/bash
#SBATCH --job-name=hist2st-eval
#SBATCH --account=YOUR_PROJECT
#SBATCH --partition=shared
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=8
#SBATCH --mem=32G
#SBATCH --time=04:00:00
#SBATCH --constraint=lustre
#SBATCH --output=logs/%x-%j.out

set -euo pipefail

PROJECT_DIR=/expanse/lustre/projects/REPLACE_WITH_ASSIGNED_PATH/hist2st
cd "$PROJECT_DIR/repo"
mkdir -p "$PROJECT_DIR/logs" "$PROJECT_DIR/runs"
export HIST2ST_DATA_DIR="$PROJECT_DIR/data"
export HIST2ST_RUN_DIR="$PROJECT_DIR/runs/eval-$SLURM_JOB_ID"
mkdir -p "$HIST2ST_RUN_DIR"

module purge
module load cpu
# Load/activate the Python environment selected in Step 6 here.
module list
python --version

jupyter nbconvert --to notebook --execute notebooks/onbd_03_pca_ridge_evaluation.ipynb \
  --output 03_pca_ridge_evaluation_$SLURM_JOB_ID.ipynb \
  --output-dir "$HIST2ST_RUN_DIR" \
  --ExecutePreprocessor.timeout=-1
```

Save this as a file such as `jobs/evaluate_cpu.sbatch`. `--time`, `--mem`, and CPU count are examples, not a promise that the full cohort fits; profile a small run and adjust. Request only the resources you expect to use. A job needing Lustre must keep the `lustre` constraint.


## 10. Submit the embedding experiment on a GPU

Notebook 02 runs the image model and is the main GPU stage. `gpu-shared` is for a single-node job using fewer than four GPUs; `gpu` is node-exclusive. The course slides' example requests one GPU, 10 CPUs, and 96 GB, but use smaller requests if a smoke test shows they are enough. One GPU in `gpu-shared` does not automatically mean you receive ample CPU or memory, so request those explicitly.

```bash
#!/bin/bash
#SBATCH --job-name=hist2st-embed
#SBATCH --account=YOUR_GPU_PROJECT
#SBATCH --partition=gpu-shared
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=10
#SBATCH --mem=96G
#SBATCH --gpus=1
#SBATCH --time=04:00:00
#SBATCH --constraint=lustre
#SBATCH --output=logs/%x-%j.out

set -euo pipefail
PROJECT_DIR=/expanse/lustre/projects/REPLACE_WITH_ASSIGNED_PATH/hist2st
cd "$PROJECT_DIR/repo"
mkdir -p "$PROJECT_DIR/logs" "$PROJECT_DIR/runs"
export HIST2ST_DATA_DIR="$PROJECT_DIR/data"
export HIST2ST_RUN_DIR="$PROJECT_DIR/runs/embeddings-$SLURM_JOB_ID"
mkdir -p "$HIST2ST_RUN_DIR"

module purge
module load gpu
# Load/activate the course-approved GPU Python environment or container here.
module list
nvidia-smi
python -c 'import torch; print("CUDA:", torch.cuda.is_available())'

jupyter nbconvert --to notebook --execute notebooks/onbd_02_genbio_embeddings.ipynb \
  --output 02_genbio_embeddings_$SLURM_JOB_ID.ipynb \
  --output-dir "$HIST2ST_RUN_DIR" \
  --ExecutePreprocessor.timeout=-1
```

Save this as `jobs/embed_gpu.sbatch`. Confirm that the Python environment contains a CUDA-enabled PyTorch build and that the model weights are available. Hugging Face authentication must work without a prompt; never put the token in this file.


## 11. Run the workflow in order

After staging data and preparing a batch-safe environment, run the stages in order. You can wait for each job to finish and submit the next one manually, or use Slurm's `afterok` dependency so a later stage starts only if the earlier stage succeeds. Create `logs/` before submitting.

```bash
mkdir -p /expanse/lustre/projects/REPLACE_WITH_ASSIGNED_PATH/hist2st/logs
sbatch jobs/prepare_cpu.sbatch
# After the preparation job succeeds, submit its dependent embedding job:
sbatch --dependency=afterok:PREPARATION_JOB_ID jobs/embed_gpu.sbatch
# After embedding succeeds, submit evaluation:
sbatch --dependency=afterok:EMBEDDING_JOB_ID jobs/evaluate_cpu.sbatch
```

This staged flow keeps CPU/GPU requests appropriate for each part and prevents evaluation from starting before its input files exist. `PREPARATION_JOB_ID` and `EMBEDDING_JOB_ID` are placeholders; use the numeric job IDs printed by `sbatch`. If you do not yet have a batch-safe preparation notebook, stage the prepared data using the course-approved data route and start at the embedding stage.


## 12. Monitor, inspect, and cancel jobs

`sbatch` returns a job ID. Use it to see whether the job is pending, running, or finished, and inspect the log created in `logs/`.

```bash
squeue -u "$USER"
scontrol show job JOB_ID
tail -f logs/hist2st-eval-JOB_ID.out
sacct -j JOB_ID --format=JobID,State,ExitCode,Elapsed,MaxRSS
```

A pending job (`PD`) is waiting for resources or a scheduling condition. A running job (`R`) is on a compute node. If a job fails, read the log and Slurm state, fix the cause, and submit a new run directory so partial output is not mistaken for a complete result. Cancel only your own job when needed:

```bash
scancel JOB_ID
```

At the end, verify expected result files and checkpoint files exist, record the job ID, configuration, random seed, module/container version, and code revision, then copy important outputs to a backed-up location. Do not assume Lustre is archival storage.


## 13. A practical first experiment

1. Confirm your ACCESS and Expanse CPU/GPU allocations and exact project path.
2. Copy the repository code, then stage a small prepared dataset in the project data directory.
3. Set `DATA_DIR_NAME` and `RUN_DIR` explicitly in the notebook settings; remove inline package installs and interactive login/Colab cells for batch execution.
4. Run a short CPU test and a one-patch GPU test using `srun`. Confirm imports, data paths, CUDA, and output writes.
5. Submit Notebook 02's full embedding run to `gpu-shared` with a unique run directory.
6. After it succeeds, submit Notebook 03 to `shared` and inspect its outputs and log.
7. Save the job IDs and environment details, and back up the useful results.

The example batch scripts are templates. The current notebooks do not yet expose a single command-line experiment entry point, so they need the batch-safety edits described above before they can be executed with `nbconvert`.


## Sources

- Course slides supplied with this request: **BDD_Expanse_Cluster_Usage_week08.pdf**, Can Ceritoglu, October 20, 2026. The slides provide the course context, basic `ssh`/`scp`, storage, Slurm, modules, and example CPU/GPU workflows.
- [Current SDSC Expanse User Guide](https://www.sdsc.edu/systems/expanse/user_guide.html) for account checks, modules, Slurm, Lustre constraints, and storage behavior.
- The current system details and limits can change. Re-check SDSC documentation and course allocation instructions before submitting jobs.
